# Synthetic Comment Generation for r/AskDocs (v2a)

This notebook generates synthetic comments for r/AskDocs OPs whose titles contain
"health" or "healthy", using Google's Gemini 2.5 Pro.

For each submission, three sets of synthetic comments are generated:

1. **75% clinician / 25% layperson**
2. **25% clinician / 75% layperson**
3. **No specified ratio**

Changes from v2:
- Prompts amended: comments should be brief; clinicians should not announce credentials;
  laypersons should not disclaim lack of training; clinicians' synthetic comments are to
  disclaimers like "it's important to speak with a real physician"
- OPs filtered to those with "health" or "healthy" in the title
- Generates for 5 randomly selected OPs
- Outputs a spreadsheet (1 tab per OP) for review

## 1. Setup & Imports

In [ ]:
# Install dependencies if needed
# !pip install google-genai tqdm openpyxl

In [1]:
import json
import os
import re
import random
import time
from getpass import getpass
from pathlib import Path
from collections import defaultdict
from tqdm import tqdm

from google import genai
from google.genai import types

In [2]:
# ---------- Configuration ----------
API_KEY = getpass("Enter your Google Gemini API key: ")
MODEL_ID = "gemini-2.5-pro"

# Paths
CORPORA_DIR = Path("../output/corpora")
SUBMISSIONS_PATH = CORPORA_DIR / "submissions_corpus.jsonl"
COMMENTS_PATH = CORPORA_DIR / "comments_corpus.jsonl"
CHECKPOINT_DIR = CORPORA_DIR / "checkpoints_2a"
OUTPUT_XLSX = CORPORA_DIR / "synthetic_comments_sample.xlsx"

CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

# Rate-limiting
DELAY_BETWEEN_CALLS = 1.0

# Number of OPs to generate for
N_SAMPLE = 5

print(f"Model: {MODEL_ID}")
print(f"Submissions: {SUBMISSIONS_PATH}")
print(f"Comments: {COMMENTS_PATH}")
print(f"Output spreadsheet: {OUTPUT_XLSX}")

Enter your Google Gemini API key:  ········


Model: gemini-2.5-pro
Submissions: ../output/corpora/submissions_corpus.jsonl
Comments: ../output/corpora/comments_corpus.jsonl
Output spreadsheet: ../output/corpora/synthetic_comments_sample.xlsx


In [3]:
# Initialize the Gemini client
client = genai.Client(api_key=API_KEY)
print("Gemini client initialized.")

Gemini client initialized.


## 2. Load Data & Filter to "health"/"healthy" OPs

In [4]:
# Load all submissions
submissions = []
with open(SUBMISSIONS_PATH) as f:
    for line in f:
        submissions.append(json.loads(line))

print(f"Loaded {len(submissions):,} total submissions")

# Build a lookup by submission id
submissions_by_id = {s["id"]: s for s in submissions}

Loaded 19,984 total submissions


In [5]:
# Load all comments and group by submission
comments_by_submission = defaultdict(list)

with open(COMMENTS_PATH) as f:
    for line in f:
        comment = json.loads(line)
        if comment.get("parent_id", "").startswith("t3_"):
            sub_id = comment["parent_id"][3:]
            comments_by_submission[sub_id].append(comment)

print(f"Found top-level comments for {len(comments_by_submission):,} submissions")
print(f"Total top-level comments: {sum(len(v) for v in comments_by_submission.values()):,}")

Found top-level comments for 19,984 submissions
Total top-level comments: 40,537


In [6]:
# Filter to OPs with "health" or "healthy" in the title
health_submissions = [
    s for s in submissions
    if re.search(r'\bhealth(y)?\b', s.get('title', ''), re.IGNORECASE)
    and len(comments_by_submission.get(s['id'], [])) >= 1
]

print(f"OPs with 'health'/'healthy' in title (with >=1 comment): {len(health_submissions):,}")
print(f"\nSample titles:")
for s in health_submissions[:5]:
    n = len(comments_by_submission.get(s['id'], []))
    print(f"  [{n} comments] {s['title']}")

OPs with 'health'/'healthy' in title (with >=1 comment): 221

Sample titles:
  [1 comments] Which medical tests to prioritize when feeling sleepy despite being healthy?
  [2 comments] Can mental health issues (e.g. depression) cause excruciating physical pain?
  [2 comments] Ongoing unexplained health issues- looking for possible next steps
  [2 comments] My 56 year old mom is getting a stent put in her heart tomorrow. She has a lot of health issues. Should I be worried?
  [4 comments] Physical sensations from holding pee, could it cause health issues?


In [7]:
# Select 5 random OPs from the filtered set
sample_submissions = random.sample(health_submissions, min(N_SAMPLE, len(health_submissions)))

print(f"Selected {len(sample_submissions)} OPs for generation:")
for s in sample_submissions:
    n = len(comments_by_submission.get(s['id'], []))
    print(f"  [{n} comments] {s['title']}")

Selected 5 OPs for generation:
  [7 comments] Can eating a lot of fruit be bad for your health ?
  [1 comments] 18yo female, 5ft3, 55kg, non smoker, GP consultation booked, photos on my page, no medications or other health issues
  [1 comments] Is eating unrefrigerated smoked salmon a serious health hazard?
  [1 comments] Incorporating healthy foods and exercise in a family that isn’t very active
  [1 comments] 27F dealing with parents health


## 3. Prompt Templates

In [8]:
PROMPT_TEMPLATES = {
    "75_clinician": (
        "The following is the opening post from a thread in the subreddit r/AskDocs. "
        "Generate {n} comments in which each comment responds to the opening post and "
        "any subsequent comments prior to it, i.e. none of the comments should be threaded. "
        "Of these comments, 75% should imitate the response of a clinician and 25% should "
        "imitate a comment from a layperson.\n\n"
        "Important guidelines for the comments:\n"
        "- Keep comments brief and natural, especially clinician responses.\n"
        "- Clinicians should NOT introduce themselves by stating their role or credentials "
        "(e.g. do not start with 'I'm a doctor' or 'As a nurse').\n"
        "- Clinicians should NOT include disclaimers in their comments like \"It's important "
        "to consult a physician\" or \"You will want to ask a doctor about this\".\n"        
        "- Laypersons should NOT disclaim their lack of medical training "
        "(e.g. do not say 'I'm not a doctor but' or 'I have no medical background').\n\n"
        "Return your response as a JSON array of objects, where each object has these fields:\n"
        "- \"body\": the text of the comment\n"
        "- \"author_type\": either \"clinician\" or \"layperson\"\n\n"
        "Opening Post:\n"
        "Title: {title}\n\n"
        "{selftext}"
    ),
    "25_clinician": (
        "The following is the opening post from a thread in the subreddit r/AskDocs. "
        "Generate {n} comments in which each comment responds to the opening post and "
        "any subsequent comments prior to it, i.e. none of the comments should be threaded. "
        "Of these comments, 25% should imitate the response of a clinician and 75% should "
        "imitate a comment from a layperson.\n\n"
        "Important guidelines for the comments:\n"
        "- Keep comments brief and natural, especially clinician responses.\n"
        "- Clinicians should NOT introduce themselves by stating their role or credentials "
        "(e.g. do not start with 'I'm a doctor' or 'As a nurse').\n"
        "- Clinicians should NOT include disclaimers in their comments like \"It's important "
        "to consult a physician\" or \"You will want to ask a doctor about this\".\n"                
        "- Laypersons should NOT disclaim their lack of medical training "
        "(e.g. do not say 'I'm not a doctor but' or 'I have no medical background').\n\n"
        "Return your response as a JSON array of objects, where each object has these fields:\n"
        "- \"body\": the text of the comment\n"
        "- \"author_type\": either \"clinician\" or \"layperson\"\n\n"
        "Opening Post:\n"
        "Title: {title}\n\n"
        "{selftext}"
    ),
    "no_ratio": (
        "The following is the opening post from a thread in the subreddit r/AskDocs. "
        "Generate {n} comments in which each comment responds to the opening post and "
        "any subsequent comments prior to it, i.e. none of the comments should be threaded.\n\n"
        "Important guidelines for the comments:\n"
        "- Keep comments brief and natural.\n"
        "- If a comment is from a clinician, they should NOT introduce themselves by stating "
        "their role or credentials (e.g. do not start with 'I'm a doctor' or 'As a nurse').\n"
        "- If a comment is from a layperson, they should NOT disclaim their lack of medical "
        "training (e.g. do not say 'I'm not a doctor but' or 'I have no medical background').\n\n"
        "Return your response as a JSON array of objects, where each object has these fields:\n"
        "- \"body\": the text of the comment\n"
        "- \"author_type\": either \"clinician\" or \"layperson\" (use your best judgment)\n\n"
        "Opening Post:\n"
        "Title: {title}\n\n"
        "{selftext}"
    ),
}

CONDITION_KEYS = list(PROMPT_TEMPLATES.keys())
print(f"Prompt conditions: {CONDITION_KEYS}")

Prompt conditions: ['75_clinician', '25_clinician', 'no_ratio']


## 4. Gemini API Call Logic

In [9]:
def call_gemini(prompt: str, retries: int = 3) -> list[dict]:
    """
    Send a prompt to Gemini and parse the JSON array response.
    Returns a list of comment dicts with 'body' and 'author_type' fields.
    """
    for attempt in range(retries):
        try:
            response = client.models.generate_content(
                model=MODEL_ID,
                contents=prompt,
                config=types.GenerateContentConfig(
                    response_mime_type="application/json",
                    temperature=1.0,
                ),
            )
            raw_text = response.text.strip()
            comments = json.loads(raw_text)

            if not isinstance(comments, list):
                raise ValueError(f"Expected a JSON array, got {type(comments).__name__}")

            validated = []
            for c in comments:
                validated.append({
                    "body": str(c.get("body", "")),
                    "author_type": str(c.get("author_type", "unknown")),
                })
            return validated

        except json.JSONDecodeError as e:
            print(f"  JSON parse error (attempt {attempt+1}/{retries}): {e}")
            if attempt < retries - 1:
                time.sleep(2 ** attempt)
        except Exception as e:
            print(f"  API error (attempt {attempt+1}/{retries}): {e}")
            if attempt < retries - 1:
                time.sleep(2 ** attempt)

    print(f"  WARNING: All {retries} retries exhausted. Returning empty list.")
    return []


def generate_synthetic_comments(submission: dict, n_comments: int) -> dict:
    results = {}
    title = submission.get("title", "")
    selftext = submission.get("selftext", "")

    for condition_key, template in PROMPT_TEMPLATES.items():
        prompt = template.format(n=n_comments, title=title, selftext=selftext)
        comments = call_gemini(prompt)
        results[condition_key] = comments
        time.sleep(DELAY_BETWEEN_CALLS)

    return results


print("API functions defined.")

API functions defined.


## 5. Checkpoint Utilities

In [10]:
CHECKPOINT_FILE = CHECKPOINT_DIR / "generation_progress_2a.jsonl"


def load_checkpoint() -> tuple[dict, set]:
    results_by_id = {}
    completed_ids = set()

    if CHECKPOINT_FILE.exists():
        with open(CHECKPOINT_FILE) as f:
            for line in f:
                record = json.loads(line)
                sub_id = record["submission_id"]
                results_by_id[sub_id] = record
                completed_ids.add(sub_id)
        print(f"Resumed from checkpoint: {len(completed_ids):,} submissions already completed.")
    else:
        print("No checkpoint found. Starting from scratch.")

    return results_by_id, completed_ids


def append_checkpoint(record: dict):
    with open(CHECKPOINT_FILE, "a") as f:
        f.write(json.dumps(record) + "\n")


print(f"Checkpoint file: {CHECKPOINT_FILE}")

Checkpoint file: ../output/corpora/checkpoints_2a/generation_progress_2a.jsonl


## 6. Generate Synthetic Comments for 5 OPs

In [11]:
results_by_id, completed_ids = load_checkpoint()

remaining = [s for s in sample_submissions if s["id"] not in completed_ids]

print(f"Selected OPs: {len(sample_submissions)}")
print(f"Already completed: {len(sample_submissions) - len(remaining)}")
print(f"Remaining: {len(remaining)}")
print(f"API calls needed: {len(remaining) * 3}")

Resumed from checkpoint: 5 submissions already completed.
Selected OPs: 5
Already completed: 0
Remaining: 5
API calls needed: 15


In [12]:
errors = []

for i, sub in enumerate(tqdm(remaining, desc="Generating synthetic comments")):
    sub_id = sub["id"]
    n_top_level = len(comments_by_submission.get(sub_id, []))

    if n_top_level == 0:
        continue

    try:
        synthetic = generate_synthetic_comments(sub, n_top_level)

        record = {
            "submission_id": sub_id,
            "n_top_level_comments": n_top_level,
            "synthetic_75_clinician": synthetic["75_clinician"],
            "synthetic_25_clinician": synthetic["25_clinician"],
            "synthetic_no_ratio": synthetic["no_ratio"],
        }

        results_by_id[sub_id] = record
        completed_ids.add(sub_id)
        append_checkpoint(record)

    except Exception as e:
        errors.append((sub_id, str(e)))
        print(f"\nError on submission {sub_id}: {e}")

print(f"\nGeneration complete. Processed {len(completed_ids):,} submissions.")
if errors:
    print(f"Errors encountered: {len(errors)}")
    for sub_id, err in errors[:10]:
        print(f"  {sub_id}: {err}")

Generating synthetic comments: 100%|██████████████| 5/5 [03:07<00:00, 37.42s/it]


Generation complete. Processed 10 submissions.


## 7. Preview Results

In [13]:
condition_labels = {
    "synthetic_75_clinician": "75% clinician / 25% layperson",
    "synthetic_25_clinician": "25% clinician / 75% layperson",
    "synthetic_no_ratio": "No specified ratio",
}

for sub in sample_submissions:
    sub_id = sub["id"]
    record = results_by_id.get(sub_id)
    if not record:
        continue

    real = comments_by_submission[sub_id]

    print(f"=== Opening Post: {sub['title']} ===\n")

    print("-- Real comments --")
    for c in sorted(real, key=lambda x: x.get("created_utc", 0))[:3]:
        if c.get("body", "") in ("[removed]", "[deleted]"):
            continue
        # "Flair" is a label r/AskDocs uses to indicate verified medical professionals
        # (e.g. "Registered Nurse", "Physician") vs. unverified users. We use it here
        # as the real-comment equivalent of the "author_type" field in synthetic comments.
        flair = c.get("author_flair_text") or "No flair"
        print(f"  [{flair}] {c['body']}\n")

    for condition, label in condition_labels.items():
        print(f"-- {label} --")
        for c in record[condition][:3]:
            print(f"  [{c['author_type']}] {c['body']}\n")

    print("\n")

=== Opening Post: Can eating a lot of fruit be bad for your health ? ===

-- Real comments --
  [Physician] that amount of fruit has nothing to do with cholesterol, but gives you vitamins, fiber and hydration. but is maybe too much for a daily dose, as fruit is still calories and sugar (dental health). and high sugar intake slows down fat loss too. also if you intake too much fiber, you can get gassy or bloating. for me, i'd cut that amount of fruit in half, but only you know how your body feels. remember to eat protein, whole grains and healthy fats too! everything in balance :)

-- 75% clinician / 25% layperson --
  [clinician] Fruit doesn't contain cholesterol. The main concern with that quantity would be the high sugar (fructose) intake, which can raise triglycerides and might not be ideal for weight loss.

  [layperson] Wow that's a lot of fruit. I did something similar and got terrible gas and bloating. Had to cut way back.

  [clinician] That's a very high fiber load all at once

## 8. Export to Spreadsheet

In [14]:
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

wb = Workbook()
# Remove the default sheet
wb.remove(wb.active)

header_font = Font(bold=True, size=11, name="Arial")
body_font = Font(size=10, name="Arial")
title_font = Font(bold=True, size=12, name="Arial")
section_fill = PatternFill('solid', fgColor='D9E1F2')  # light blue
wrap_alignment = Alignment(wrap_text=True, vertical='top')
thin_border = Border(
    bottom=Side(style='thin', color='CCCCCC')
)

condition_labels_xlsx = {
    "synthetic_75_clinician": "Synthetic: 75% clinician / 25% layperson",
    "synthetic_25_clinician": "Synthetic: 25% clinician / 75% layperson",
    "synthetic_no_ratio": "Synthetic: No specified ratio",
}

for idx, sub in enumerate(sample_submissions):
    sub_id = sub["id"]
    record = results_by_id.get(sub_id)
    if not record:
        continue

    real = comments_by_submission.get(sub_id, [])
    real_sorted = sorted(real, key=lambda x: x.get("created_utc", 0))

    # Create a sheet with a short name (Excel limits to 31 chars)
    short_title = sub.get("title", f"OP {idx+1}")[:28]
    sheet_name = f"{idx+1}. {short_title}"
    if len(sheet_name) > 31:
        sheet_name = sheet_name[:31]
    ws = wb.create_sheet(title=sheet_name)

    # Column widths
    ws.column_dimensions['A'].width = 15
    ws.column_dimensions['B'].width = 90

    row = 1

    # ---- Opening Post ----
    ws.cell(row=row, column=1, value="Opening Post").font = title_font
    row += 1
    ws.cell(row=row, column=1, value="Title:").font = header_font
    ws.cell(row=row, column=2, value=sub.get("title", "")).font = body_font
    ws.cell(row=row, column=2).alignment = wrap_alignment
    row += 1
    ws.cell(row=row, column=1, value="Body:").font = header_font
    ws.cell(row=row, column=2, value=sub.get("selftext", "")).font = body_font
    ws.cell(row=row, column=2).alignment = wrap_alignment
    row += 2

    # ---- Real Comments ----
    ws.cell(row=row, column=1, value="Real Comments").font = title_font
    ws.cell(row=row, column=1).fill = section_fill
    ws.cell(row=row, column=2).fill = section_fill
    row += 1
    ws.cell(row=row, column=1, value="Author Type").font = header_font
    ws.cell(row=row, column=2, value="Comment").font = header_font
    row += 1
    for c in real_sorted:
        if c.get("body", "") in ("[removed]", "[deleted]"):
            continue
        # "author_flair_text" is the r/AskDocs flair indicating whether the commenter
        # is a verified clinician (e.g. "Physician") or an unverified user. This is
        # the real-data analog of the "author_type" label in the synthetic comments.
        flair = c.get("author_flair_text") or "No flair"
        ws.cell(row=row, column=1, value=flair).font = body_font
        flair = c.get("author_flair_text") or "No flair"
        ws.cell(row=row, column=1, value=flair).font = body_font
        ws.cell(row=row, column=1).alignment = wrap_alignment
        ws.cell(row=row, column=2, value=c.get("body", "")).font = body_font
        ws.cell(row=row, column=2).alignment = wrap_alignment
        ws.cell(row=row, column=1).border = thin_border
        ws.cell(row=row, column=2).border = thin_border
        row += 1
    row += 1

    # ---- Synthetic Comment Conditions ----
    for condition_key, label in condition_labels_xlsx.items():
        ws.cell(row=row, column=1, value=label).font = title_font
        ws.cell(row=row, column=1).fill = section_fill
        ws.cell(row=row, column=2).fill = section_fill
        row += 1
        ws.cell(row=row, column=1, value="Author Type").font = header_font
        ws.cell(row=row, column=2, value="Comment").font = header_font
        row += 1
        for c in record[condition_key]:
            ws.cell(row=row, column=1, value=c["author_type"]).font = body_font
            ws.cell(row=row, column=1).alignment = wrap_alignment
            ws.cell(row=row, column=2, value=c["body"]).font = body_font
            ws.cell(row=row, column=2).alignment = wrap_alignment
            ws.cell(row=row, column=1).border = thin_border
            ws.cell(row=row, column=2).border = thin_border
            row += 1
        row += 1

wb.save(OUTPUT_XLSX)
print(f"Spreadsheet saved to {OUTPUT_XLSX}")
print(f"Tabs: {wb.sheetnames}")

Spreadsheet saved to ../output/corpora/synthetic_comments_sample.xlsx
Tabs: ['1. Can eating a lot of fruit be', '2. 18yo female, 5ft3, 55kg, non', '3. Is eating unrefrigerated smo', '4. Incorporating healthy foods ', '5. 27F dealing with parents hea']
